Convert Yak model.pt into a Hugging Face-compatible model
This notebook targets the classifier checkpoint from the feature/finetune branch of gowthambalachandhiran/Yak.

It preserves Yak's original inference design:
1. Encode the input text (or SNLI premise + hypothesis).
2. Append candidate label names to the same sequence.
3. Run the Yak encoder.
4. Gather the hidden state at the first token of each candidate label.
5. Apply the shared Linear(128, 1) scorer and softmax over candidates.
The exported model is a Hugging Face PreTrainedModel with save_pretrained() / from_pretrained() support. It is not converted into a fixed-output AutoModelForSequenceClassification, because that would change Yak's candidate-label architecture.

Run this notebook from the Yak repository root, or set REPO_DIR below. The checkpoint is expected at runs/model.pt by default; edit CHECKPOINT_PATH if yours is elsewhere.

In [1]:
# 1. Locate the repository and install dependencies if needed
from pathlib import Path
import os, sys, json, shutil, re, math

# EDIT THIS if the notebook is not launched from inside the Yak repository.
REPO_DIR = Path.cwd()

def find_repo(start: Path):
    for p in [start, *start.parents]:
        if (p / "Model" / "MlmModel.py").exists() and (p / "Tokenizer" / "Tokenizer.py").exists():
            return p
    raise FileNotFoundError(
        "Could not find the Yak repository. Set REPO_DIR to the folder containing Model/ and Tokenizer/."
    )

REPO_DIR = find_repo(REPO_DIR)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Yak repository:", REPO_DIR)


Yak repository: c:\Users\USER\Downloads\Our Jev\Yak


In [4]:
# 2. Configure checkpoint, tokenizer, metadata, and export paths
CHECKPOINT_PATH = REPO_DIR / "runs" /"mixture" /"best.pt"
TOKENIZER_PATH = REPO_DIR / "Tokenizer" / "TokenFile" / "bpe.json"
LABEL_METADATA_PATH = REPO_DIR / "Data" / "processed" / "label_metadata.json"
EXPORT_DIR = REPO_DIR / "runs" / "huggingface_yak_classifier"

# If your checkpoint is stored elsewhere, replace CHECKPOINT_PATH, for example:
# CHECKPOINT_PATH = REPO_DIR / ".Yak" / "runs" / "model.pt"

for name, path in {
    "checkpoint": CHECKPOINT_PATH,
    "tokenizer": TOKENIZER_PATH,
    "label metadata": LABEL_METADATA_PATH,
}.items():
    print(f"{name:16s}: {path} | exists={path.exists()}")

assert CHECKPOINT_PATH.exists(), f"Checkpoint not found: {CHECKPOINT_PATH}"
assert TOKENIZER_PATH.exists(), f"BPE tokenizer file not found: {TOKENIZER_PATH}"
assert LABEL_METADATA_PATH.exists(), f"Label metadata not found: {LABEL_METADATA_PATH}"


checkpoint      : c:\Users\USER\Downloads\Our Jev\Yak\runs\mixture\best.pt | exists=True
tokenizer       : c:\Users\USER\Downloads\Our Jev\Yak\Tokenizer\TokenFile\bpe.json | exists=True
label metadata  : c:\Users\USER\Downloads\Our Jev\Yak\Data\processed\label_metadata.json | exists=True


In [5]:
# 3. Install/import Hugging Face dependencies
%pip install -q -U transformers safetensors

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import PretrainedConfig, PreTrainedModel
from transformers.modeling_outputs import SequenceClassifierOutput

from Model.LayerOne.EmbedPostional import EmbedPos
from Model.EncoderBlock import EncoderBlock
from Tokenizer.Tokenizer import BPE
from Model.OurExperiments.FineTuning.Utils.FastBpe import FastBPE, normalize_label

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", device)


Note: you may need to restart the kernel to use updated packages.


c:\Users\USER\anaconda3\envs\reasoning\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1115 14:44:47.090000 25712 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


PyTorch: 2.10.0+cpu
Device: cpu


In [6]:
# 4. Load Yak's custom BPE tokenizer and inspect its special-token IDs
bpe = BPE()
bpe.load(str(TOKENIZER_PATH))
fast_bpe = FastBPE(bpe)

print("Vocabulary size:", fast_bpe.vocab_size)
print("Special tokens:", bpe.special_tokens)
print({
    "pad": fast_bpe.pad_id,
    "unk": fast_bpe.unk_id,
    "cls": fast_bpe.cls_id,
    "sep": fast_bpe.sep_id,
    "mask": fast_bpe.mask_id,
})

assert fast_bpe.vocab_size == 30005, (
    f"Expected the 30,005-token Yak vocabulary described by this branch; got {fast_bpe.vocab_size}. "
    "Check that TOKENIZER_PATH points to the matching tokenizer checkpoint."
)


Vocabulary size: 30005
Special tokens: ['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]']
{'pad': 0, 'unk': 1, 'cls': 2, 'sep': 3, 'mask': 4}


5. Define the Hugging Face model wrapper
The parameter names deliberately match the repository's YakClassifier (embed, blocks, final_norm, scorer). This allows the classifier checkpoint to load without changing its learned weights.

In [21]:

import torch
import torch.nn as nn
from transformers import PretrainedConfig, PreTrainedModel


class YakHFConfig(PretrainedConfig):

    model_type = "yak-candidate-classifier"

    def __init__(
        self,
        vocab_size=30005,
        d_model=128,
        nhead=4,
        num_layers=4,
        max_seq_len=2048,
        dropout=0.1,
        pad_token_id=0,
        **kwargs,
    ):
        super().__init__(
            pad_token_id=pad_token_id,
            **kwargs
        )

        self.vocab_size = vocab_size
        self.d_model = d_model
        self.nhead = nhead
        self.num_layers = num_layers
        self.max_seq_len = max_seq_len
        self.dropout = dropout

        # Keep aliases for compatibility with the original Yak code
        self.layers = num_layers
        self.max_seq = max_seq_len
        self.pad_id = pad_token_id


class YakHFForCandidateClassification(PreTrainedModel):

    config_class = YakHFConfig
    base_model_prefix = "yak"

    # Yak does not use tied weights
    all_tied_weights_keys = {}

    def __init__(self, config):

        super().__init__(config)

        self.embed = EmbedPos(
            config.vocab_size,
            config.d_model,
            config.max_seq_len,
            config.pad_token_id,
            config.dropout,
        )

        self.blocks = nn.ModuleList([
            EncoderBlock(
                config.d_model,
                config.nhead,
                config.dropout,
            )
            for _ in range(config.num_layers)
        ])

        self.final_norm = nn.LayerNorm(config.d_model)

        self.scorer = nn.Linear(config.d_model, 1)

        self.post_init()

    
    def forward(
        self,
        input_ids=None,
        label_positions=None,
        label_mask=None,
        attention_mask=None,
        **kwargs,
    ):
        if input_ids is None:
            raise ValueError("input_ids must be provided")

        if label_positions is None:
            raise ValueError("label_positions must be provided")

        if label_mask is None:
            raise ValueError("label_mask must be provided")

        # Identify non-padding tokens
        keep = input_ids != self.config.pad_token_id

        # Yak's original encoder uses True for valid tokens
        padding_mask = None if bool(keep.all()) else keep

        # Embedding
        x = self.embed(input_ids)

        # Transformer encoder blocks
        for block in self.blocks:
            x = block(x, padding_mask)

        # Final normalization
        x = self.final_norm(x)

        # Gather hidden states at candidate-label positions
        batch_size = x.size(0)
        batch_indices = torch.arange(
            batch_size,
            device=x.device
        ).unsqueeze(1)

        candidate_hidden = x[batch_indices, label_positions]

        # Shared candidate scorer
        scores = self.scorer(candidate_hidden).squeeze(-1)

        # Mask unused candidate positions
        scores = scores.masked_fill(~label_mask.bool(), float("-inf"))

        return scores



# Instantiate model and inspect parameter count
config = YakHFConfig()

model = YakHFForCandidateClassification(config)

print("Model initialized successfully!")
print("Model parameters:", sum(p.numel() for p in model.parameters()))
print("Vocabulary size:", config.vocab_size)
print("Number of layers:", config.num_layers)
print("Maximum sequence length:", config.max_seq_len)


Model initialized successfully!
Model parameters: 4634113
Vocabulary size: 30005
Number of layers: 4
Maximum sequence length: 2048


6. Load model.pt
This loader accepts common checkpoint containers (state_dict, model_state_dict, model, etc.) and strips common wrappers such as module.. It then checks that the checkpoint contains the classifier scorer as well as the encoder. A phase-1 MLM-only checkpoint is not sufficient for this notebook.

In [22]:
def extract_state_dict(checkpoint_obj):
    obj = checkpoint_obj
    if not isinstance(obj, dict):
        raise TypeError(f"Expected a dict-like checkpoint, got {type(obj)}")

    # Common training-checkpoint wrappers.
    for key in ("model_state_dict", "model_state", "state_dict", "model", "classifier"):
        if key in obj and isinstance(obj[key], dict):
            obj = obj[key]
            break

    if not isinstance(obj, dict) or not obj:
        raise ValueError("Could not find a non-empty state_dict in the checkpoint.")

    # Remove wrappers only when they prefix every key.
    state = dict(obj)
    prefixes = ("module.", "_orig_mod.", "model.", "classifier.")
    changed = True
    while changed:
        changed = False
        for prefix in prefixes:
            if state and all(k.startswith(prefix) for k in state):
                state = {k[len(prefix):]: v for k, v in state.items()}
                changed = True
                break
    return state


try:
    raw_checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
except TypeError:
    raw_checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")

state_dict = extract_state_dict(raw_checkpoint)

# Keep only parameters belonging to YakClassifier.
expected_keys = set(model.state_dict().keys())
filtered_state = {k: v for k, v in state_dict.items() if k in expected_keys}

missing = sorted(expected_keys - set(filtered_state))
unexpected_model_keys = sorted(k for k in state_dict if k.startswith(
    ("embed.", "blocks.", "final_norm.", "scorer.")
) and k not in expected_keys)

print("Checkpoint tensors:", len(state_dict))
print("Matching model tensors:", len(filtered_state))
print("Missing model tensors:", missing[:20])
print("Unexpected Yak tensors:", unexpected_model_keys[:20])

if not any(k.startswith("scorer.") for k in filtered_state):
    raise RuntimeError(
        "No scorer.* weights were found. This appears to be an MLM/pretraining checkpoint, "
        "not the classifier checkpoint. Use the classifier model.pt/best.pt."
    )

if missing or unexpected_model_keys:
    raise RuntimeError(
        "Checkpoint does not exactly match the classifier architecture. "
        f"Missing={missing[:20]}, unexpected={unexpected_model_keys[:20]}"
    )

load_result = model.load_state_dict(filtered_state, strict=True)
model.to(device).eval()
print("Loaded classifier checkpoint successfully:", load_result)


Checkpoint tensors: 53
Matching model tensors: 53
Missing model tensors: []
Unexpected Yak tensors: []
Loaded classifier checkpoint successfully: <All keys matched successfully>


In [23]:
def extract_state_dict(checkpoint_obj):
    obj = checkpoint_obj
    if not isinstance(obj, dict):
        raise TypeError(f"Expected a dict-like checkpoint, got {type(obj)}")

    # Common training-checkpoint wrappers.
    for key in ("model_state_dict", "model_state", "state_dict", "model", "classifier"):
        if key in obj and isinstance(obj[key], dict):
            obj = obj[key]
            break

    if not isinstance(obj, dict) or not obj:
        raise ValueError("Could not find a non-empty state_dict in the checkpoint.")

    # Remove wrappers only when they prefix every key.
    state = dict(obj)
    prefixes = ("module.", "_orig_mod.", "model.", "classifier.")
    changed = True
    while changed:
        changed = False
        for prefix in prefixes:
            if state and all(k.startswith(prefix) for k in state):
                state = {k[len(prefix):]: v for k, v in state.items()}
                changed = True
                break
    return state


try:
    raw_checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
except TypeError:
    raw_checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")

state_dict = extract_state_dict(raw_checkpoint)

# Keep only parameters belonging to YakClassifier.
expected_keys = set(model.state_dict().keys())
filtered_state = {k: v for k, v in state_dict.items() if k in expected_keys}

missing = sorted(expected_keys - set(filtered_state))
unexpected_model_keys = sorted(k for k in state_dict if k.startswith(
    ("embed.", "blocks.", "final_norm.", "scorer.")
) and k not in expected_keys)

print("Checkpoint tensors:", len(state_dict))
print("Matching model tensors:", len(filtered_state))
print("Missing model tensors:", missing[:20])
print("Unexpected Yak tensors:", unexpected_model_keys[:20])

if not any(k.startswith("scorer.") for k in filtered_state):
    raise RuntimeError(
        "No scorer.* weights were found. This appears to be an MLM/pretraining checkpoint, "
        "not the classifier checkpoint. Use the classifier model.pt/best.pt."
    )

if missing or unexpected_model_keys:
    raise RuntimeError(
        "Checkpoint does not exactly match the classifier architecture. "
        f"Missing={missing[:20]}, unexpected={unexpected_model_keys[:20]}"
    )

load_result = model.load_state_dict(filtered_state, strict=True)
model.to(device).eval()
print("Loaded classifier checkpoint successfully:", load_result)


Checkpoint tensors: 53
Matching model tensors: 53
Missing model tensors: []
Unexpected Yak tensors: []
Loaded classifier checkpoint successfully: <All keys matched successfully>


7. Export in Hugging Face save_pretrained() format
The export includes the model weights/configuration, the original Yak BPE merges, and the task label metadata. Keep the tokenizer and model together: the custom BPE vocabulary must match the embedding matrix.

In [24]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(EXPORT_DIR, safe_serialization=True)
shutil.copy2(TOKENIZER_PATH, EXPORT_DIR / "bpe.json")
shutil.copy2(LABEL_METADATA_PATH, EXPORT_DIR / "label_metadata.json")

# A small manifest records the custom inference contract.
manifest = {
    "model_type": "yak-candidate-classifier",
    "architecture": "YakClassifier",
    "candidate_label_scoring": "shared Linear(d_model, 1) over first-token hidden states",
    "tokenizer": "Yak custom BPE; use bpe.json and the inference helper in this notebook",
    "tasks": ["banking77", "clinc", "snli"],
    "max_seq_len": config.max_seq_len,
    "max_text_tokens": 96,
    "max_premise_tokens": 64,
    "max_hypothesis_tokens": 32,
    "max_label_tokens": 12,
}
(EXPORT_DIR / "yak_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("Exported to:", EXPORT_DIR)
print("Files:", sorted(p.name for p in EXPORT_DIR.iterdir()))


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 103.08it/s]

Exported to: c:\Users\USER\Downloads\Our Jev\Yak\runs\huggingface_yak_classifier
Files: ['bpe.json', 'config.json', 'label_metadata.json', 'model.safetensors', 'yak_manifest.json']


8. Build candidate-label inputs exactly as in Yak
For ordinary classification, the packed sequence is:

[CLS] text [SEP] label_1 [SEP] label_2 [SEP] ...

For SNLI, Yak joins the two sentences with a separator before appending candidate labels. The label order is preserved in the returned probability list.

In [29]:
LABEL_ALIASES = {"oos": "out of scope"}

def encode_text_for_task(text=None, premise=None, hypothesis=None,
                         max_text_tokens=96, max_premise_tokens=64, max_hypothesis_tokens=32):
    if premise is not None or hypothesis is not None:
        if premise is None or hypothesis is None:
            raise ValueError("Provide both premise and hypothesis for SNLI.")
        p_ids = fast_bpe.encode_ids(premise, normalize=True)[:max_premise_tokens]
        h_ids = fast_bpe.encode_ids(hypothesis, normalize=True)[:max_hypothesis_tokens]
        return p_ids + [fast_bpe.sep_id] + h_ids
    if text is None:
        raise ValueError("Provide text, or both premise and hypothesis.")
    return fast_bpe.encode_ids(text, normalize=True)[:max_text_tokens]


def pack_candidate_labels(text_ids, candidate_labels, max_label_tokens=12):
    if not candidate_labels:
        raise ValueError("candidate_labels must contain at least one label.")

    ids = [fast_bpe.cls_id] + list(text_ids) + [fast_bpe.sep_id]
    label_positions = []

    for label in candidate_labels:
        label_text = LABEL_ALIASES.get(label, normalize_label(label))
        label_ids = fast_bpe.encode_ids(label_text, normalize=True)[:max_label_tokens]
        if not label_ids:
            label_ids = [fast_bpe.unk_id]
        label_positions.append(len(ids))
        ids.extend(label_ids)
        ids.append(fast_bpe.sep_id)

    if len(ids) > config.max_seq_len:
        raise ValueError(
            f"Packed input has {len(ids)} tokens, exceeding max_seq_len={config.max_seq_len}. "
            "Reduce the text length or candidate-label count."
        )

    input_ids = torch.tensor([ids], dtype=torch.long, device=device)
    label_positions = torch.tensor([label_positions], dtype=torch.long, device=device)
    label_mask = torch.ones_like(label_positions, dtype=torch.bool, device=device)
    return input_ids, label_positions, label_mask


@torch.inference_mode()
def predict_candidates(candidate_labels, text=None, premise=None, hypothesis=None):
    model.eval()
    text_ids = encode_text_for_task(text=text, premise=premise, hypothesis=hypothesis)
    input_ids, label_positions, label_mask = pack_candidate_labels(text_ids, candidate_labels)
    outputs = model(
        input_ids=input_ids,
        label_positions=label_positions,
        label_mask=label_mask,
    )
    
    probabilities = torch.softmax(outputs.float(), dim=-1)[0].cpu().tolist()
    ranked = sorted(
        [{"label": label, "score": float(score)} for label, score in zip(candidate_labels, probabilities)],
        key=lambda item: item["score"],
        reverse=True,
    )
    return ranked


9. Load the exported model back from disk
This verifies that the Hugging Face checkpoint can be reconstructed independently of the in-memory model used during conversion.

In [30]:
reloaded_config = YakHFConfig.from_pretrained(EXPORT_DIR)
reloaded_model = YakHFForCandidateClassification.from_pretrained(
    EXPORT_DIR,
    config=reloaded_config,
).to(device).eval()

print("Reloaded model:", type(reloaded_model).__name__)
print("Reloaded parameters:", sum(p.numel() for p in reloaded_model.parameters()))


Loading weights: 100%|██████████| 53/53 [00:00<00:00, 3147.67it/s]

Reloaded model: YakHFForCandidateClassification
Reloaded parameters: 4634113


10. Run inference
The metadata file contains the label vocabularies for all three tasks. Select the relevant task and pass its candidate labels. For SNLI, pass both premise and hypothesis.

In [32]:
label_metadata = json.loads(LABEL_METADATA_PATH.read_text(encoding="utf-8"))
label_vocabularies = label_metadata["label_vocabularies"]

print("Available tasks:", list(label_vocabularies))
for task_name, names in label_vocabularies.items():
    print(f"{task_name:10s}: {len(names)} candidate labels")

# Example: SNLI inference
snli_labels = label_vocabularies["snli"]
snli_result = predict_candidates(
    candidate_labels=snli_labels,
    premise="A person is riding a bicycle down the street.",
    hypothesis="Someone is outdoors.",
)
snli_result[:6]


Available tasks: ['snli', 'banking77', 'clinc']
snli      : 3 candidate labels
banking77 : 77 candidate labels
clinc     : 151 candidate labels


[{'label': 'entailment', 'score': 0.9278973937034607},
 {'label': 'contradiction', 'score': 0.039467014372348785},
 {'label': 'neutral', 'score': 0.03263559937477112}]

In [34]:
# Example: Banking77 inference
banking_labels = label_vocabularies["banking77"]
banking_result = predict_candidates(
    candidate_labels=banking_labels,
    text="My card has not arrived yet and I want to know when it will be delivered.",
)
banking_result[:5]


[{'label': 'card_not_working', 'score': 0.14252935349941254},
 {'label': 'card_linking', 'score': 0.1331583857536316},
 {'label': 'card_about_to_expire', 'score': 0.1308031678199768},
 {'label': 'card_arrival', 'score': 0.12383675575256348},
 {'label': 'card_delivery_estimate', 'score': 0.1076035276055336}]

In [36]:
# Example: CLINC inference
clinc_labels = label_vocabularies["clinc"]
clinc_result = predict_candidates(
    candidate_labels=clinc_labels,
    text="I want to block my ATM pin how do I do that?",
)
clinc_result[:5]


[{'label': 'pin_change', 'score': 0.8380526900291443},
 {'label': 'jump_start', 'score': 0.07065417617559433},
 {'label': 'roll_dice', 'score': 0.04673239216208458},
 {'label': 'plug_type', 'score': 0.014525173231959343},
 {'label': 'tire_change', 'score': 0.0035190500784665346}]

11. Check numerical equivalence against the original loaded model
The exported and reloaded model should produce the same logits as the in-memory model for an identical packed input. This is a useful conversion sanity check; it does not measure task accurac

In [39]:

import torch

model.eval()
reloaded_model.eval()

original_state = model.state_dict()
reloaded_state = reloaded_model.state_dict()

print("Original model parameters:", sum(p.numel() for p in model.parameters()))
print("Reloaded model parameters:", sum(p.numel() for p in reloaded_model.parameters()))

print("\nOriginal state keys:", len(original_state))
print("Reloaded state keys:", len(reloaded_state))

original_keys = set(original_state.keys())
reloaded_keys = set(reloaded_state.keys())

print("\nMissing keys in reloaded model:")
print(original_keys - reloaded_keys)

print("\nUnexpected keys in reloaded model:")
print(reloaded_keys - original_keys)

# Compare matching tensors
differences = []

for key in sorted(original_keys & reloaded_keys):

    a = original_state[key].detach().cpu()
    b = reloaded_state[key].detach().cpu()

    if a.shape != b.shape:
        print(f"SHAPE MISMATCH: {key}: {a.shape} vs {b.shape}")
        continue

    if a.is_floating_point():
        diff = (a.float() - b.float()).abs().max().item()
    else:
        diff = 0.0 if torch.equal(a, b) else float("inf")

    differences.append((key, diff))

print("\nLargest parameter differences:")

for key, diff in sorted(
    differences,
    key=lambda x: x[1],
    reverse=True
)[:15]:
    print(f"{key:45s} {diff:.8f}")

print("\nIdentical state dictionaries:",
      original_keys == reloaded_keys and
      all(diff == 0.0 for _, diff in differences))


Original model parameters: 4634113
Reloaded model parameters: 4634113

Original state keys: 53
Reloaded state keys: 53

Missing keys in reloaded model:
set()

Unexpected keys in reloaded model:
set()

Largest parameter differences:
blocks.0.attention.out_proj.bias              0.00000000
blocks.0.attention.out_proj.weight            0.00000000
blocks.0.attention.qkv_proj.bias              0.00000000
blocks.0.attention.qkv_proj.weight            0.00000000
blocks.0.ffn.linear1.bias                     0.00000000
blocks.0.ffn.linear1.weight                   0.00000000
blocks.0.ffn.linear2.bias                     0.00000000
blocks.0.ffn.linear2.weight                   0.00000000
blocks.0.norm1.bias                           0.00000000
blocks.0.norm1.weight                         0.00000000
blocks.0.norm2.bias                           0.00000000
blocks.0.norm2.weight                         0.00000000
blocks.1.attention.out_proj.bias              0.00000000
blocks.1.attention.out_proj

In [42]:

import torch

model.eval()
reloaded_model.eval()

# 1. Prepare a manual SNLI example
text_ids = encode_text_for_task(
    premise="A person is riding a bicycle down the street.",
    hypothesis="Someone is outdoors.",
)

# 2. Pack candidate labels
input_ids, label_positions, label_mask = pack_candidate_labels(
    text_ids,
    snli_labels,
)

# 3. Move inputs to the appropriate device
input_ids = input_ids.to(device)
label_positions = label_positions.to(device)
label_mask = label_mask.to(device)

# 4. Compare the inputs
print("Input IDs shape:", input_ids.shape)
print("Label positions shape:", label_positions.shape)
print("Label mask shape:", label_mask.shape)

# 5. Compare intermediate outputs
@torch.inference_mode()
def get_intermediates(m):
    x = m.embed(input_ids)

    outputs = {"embedding": x.detach().clone()}

    keep = input_ids != m.config.pad_token_id
    padding_mask = None if bool(keep.all()) else keep

    for i, block in enumerate(m.blocks):
        x = block(x, padding_mask)
        outputs[f"block_{i}"] = x.detach().clone()

    x = m.final_norm(x)
    outputs["final_norm"] = x.detach().clone()

    batch_indices = torch.arange(
        x.size(0), device=x.device
    ).unsqueeze(1)

    candidate_hidden = x[batch_indices, label_positions]

    outputs["candidate_hidden"] = candidate_hidden.detach().clone()

    outputs["scores"] = m.scorer(candidate_hidden).squeeze(-1)

    return outputs


original_outputs = get_intermediates(model)
reloaded_outputs = get_intermediates(reloaded_model)

print("\nIntermediate output comparison:")

for name in original_outputs:
    diff = (
        original_outputs[name].float()
        - reloaded_outputs[name].float()
    ).abs().max().item()

    print(f"{name:25s}: {diff:.10f}")


Input IDs shape: torch.Size([1, 28])
Label positions shape: torch.Size([1, 3])
Label mask shape: torch.Size([1, 3])

Intermediate output comparison:
embedding                : 1.0000001192
block_0                  : 6.3154382706
block_1                  : 6.1613845825
block_2                  : 7.0358352661
block_3                  : 15.6221103668
final_norm               : 8.0112504959
candidate_hidden         : 5.8740701675
scores                   : 2.1661067009


In [41]:
@torch.inference_mode()
def compare_intermediate_outputs(input_ids, label_positions):

    def get_intermediates(m):
        x = m.embed(input_ids)

        outputs = {"embedding": x.detach().clone()}

        keep = input_ids != m.config.pad_token_id
        padding_mask = None if bool(keep.all()) else keep

        for i, block in enumerate(m.blocks):
            x = block(x, padding_mask)
            outputs[f"block_{i}"] = x.detach().clone()

        x = m.final_norm(x)
        outputs["final_norm"] = x.detach().clone()

        batch_indices = torch.arange(
            x.size(0), device=x.device
        ).unsqueeze(1)

        candidate_hidden = x[batch_indices, label_positions]
        outputs["candidate_hidden"] = candidate_hidden.detach().clone()

        outputs["scores"] = m.scorer(candidate_hidden).squeeze(-1)

        return outputs

    original = get_intermediates(model)
    reloaded = get_intermediates(reloaded_model)

    for name in original:
        diff = (
            original[name].float() - reloaded[name].float()
        ).abs().max().item()

        print(f"{name:25s}: {diff:.10f}")


text_ids = encode_text_for_task(
    premise="A person is riding a bicycle down the street.",
    hypothesis="Someone is outdoors.",
)

input_ids, label_positions, label_mask = pack_candidate_labels(
    text_ids, snli_labels
)

compare_intermediate_outputs(input_ids, label_positions)

embedding                : 1.0000001192
block_0                  : 6.3154382706
block_1                  : 6.1613845825
block_2                  : 7.0358352661
block_3                  : 15.6221103668
final_norm               : 8.0112504959
candidate_hidden         : 5.8740701675
scores                   : 2.1661067009


In [38]:

@torch.inference_mode()
def compare_original_and_reloaded(
    candidate_labels,
    text=None,
    premise=None,
    hypothesis=None,
):
    text_ids = encode_text_for_task(
        text=text,
        premise=premise,
        hypothesis=hypothesis,
    )

    input_ids, label_positions, label_mask = pack_candidate_labels(
        text_ids,
        candidate_labels,
    )

    # Original model
    original_logits = model(
        input_ids=input_ids,
        label_positions=label_positions,
        label_mask=label_mask,
    )

    # Reloaded Hugging Face model
    reloaded_logits = reloaded_model(
        input_ids=input_ids,
        label_positions=label_positions,
        label_mask=label_mask,
    )

    # Compare raw candidate scores
    max_abs_diff = (
        original_logits - reloaded_logits
    ).abs().max().item()

    print("Maximum absolute logit difference:", max_abs_diff)

    assert torch.allclose(
        original_logits,
        reloaded_logits,
        atol=1e-6,
        rtol=1e-5,
    ), "Mismatch between original and reloaded model!"

    print("PASS: Exported model reproduces the original model scores.")


compare_original_and_reloaded(
    candidate_labels=snli_labels,
    premise="A person is riding a bicycle down the street.",
    hypothesis="Someone is outdoors.",
)


Maximum absolute logit difference: 2.166106700897217


AssertionError: Mismatch between original and reloaded model!